In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Upload data from csv file



In [2]:
df = pd.read_csv("/content/insurance.csv")
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


##Clean and Encode Non-numerical Values



In [3]:
df_cleaned = df.dropna()
df_encoded = pd.get_dummies(df_cleaned, columns=[
    "sex",
    "smoker",
    "region",
], drop_first=True)

In [4]:
#store input columns in X
X = df_encoded[['age','bmi','children','sex_male','smoker_yes','region_northwest','region_southeast','region_southwest']]

In [5]:
#store output values in Y
Y = df_encoded["charges"]

##Split data into training and testing sections

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=0.2, random_state=100)

##Create Random Forest Regressor model and fit training data

In [7]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,random_state=42
)
rf.fit(x_train,y_train)


RandomForestRegressor(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=300, random_state=42)

##Use model to predict outcomes of training set and testing set

In [8]:
y_rf_train_pred = rf.predict(x_train)
y_rf_test_pred = rf.predict(x_test)

##Compare values in a dataframe showing difference

In [9]:
compare_outcomes = np.column_stack((y_train, y_rf_train_pred))
results = pd.DataFrame(compare_outcomes, columns=['actual_outcome','predicted_outcome'])
results['difference'] = abs(results['predicted_outcome'] - results['actual_outcome'])
results

,actual_outcome,predicted_outcome,difference
0,16115.30450,16369.317284,254.012784
1,10115.00885,10987.655699,872.646849
2,13635.63790,13485.238792,150.399108
3,5836.52040,6770.884360,934.363960
4,8871.15170,9701.126314,829.974614
...,...,...,...
1065,2103.08000,2339.244549,236.164549
1066,37742.57570,37963.725849,221.150149
1067,11830.60720,11983.700722,153.093522
1068,6571.02435,6456.807449,114.216901


##Calculate RMSE and r^2 values to determine accuracy of model

In [10]:
from sklearn.metrics import mean_squared_error, r2_score

rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_rf_train_pred))
rf_train_r2 = r2_score(y_train, y_rf_train_pred)

rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_rf_test_pred))
rf_test_r2 = r2_score(y_test, y_rf_test_pred)

##Store statistical results in dataframe

In [11]:
result = pd.DataFrame([rf_train_rmse, rf_train_r2, rf_test_rmse, rf_test_r2]).transpose()
result.columns = ["Training RMSE", "Training R^2", "Testing RMSE", "Testing R^2"]
result

,Training RMSE,Training R^2,Testing RMSE,Testing R^2
0,3121.700514,0.932214,4119.11368,0.891797


###Give new data to make a prediction

In [12]:
new_data = pd.read_csv("/content/new_data.csv")

##Predict value of charges for new data and export to CSV file

In [13]:
predicted_charges = rf.predict(new_data)

new_data['predicted_charges'] = predicted_charges
new_data.to_csv("/content/predicted_charges.csv", index=False)

In [14]:
new_data

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,predicted_charges
0,23,27.5,0,1,0,0,1,0,4157.520258
1,45,28.1,2,0,1,1,0,0,25059.980990
2,31,22.8,1,1,0,0,0,1,5286.121461
3,54,35.6,3,0,1,0,1,0,45010.462243
4,19,24.1,0,1,0,1,0,0,1653.054521
5,37,29.4,2,0,0,0,0,1,7119.799326
6,62,33.8,1,1,1,0,1,0,47400.588307
7,28,26.3,0,0,0,1,0,0,3961.037169
8,41,31.9,2,1,0,0,1,0,8472.377751
9,50,28.7,3,0,1,0,0,1,25286.176276
